# 11. AIND Units NWB Export

Build an `AINDUnitsNWBScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDUnitsNWBTask` for each coordinate. The task itself clones [aind-units-nwb](https://github.com/AllenNeuralDynamics/aind-units-nwb) on first use, patches `utils.py` for two neuroconv API renames (`add_electrodes_info_to_nwbfile` → `add_electrodes_to_nwbfile`, `add_units_table_to_nwbfile` → `_add_units_table_to_nwbfile`), seeds its `data/` with the base NWB + `preprocessed/` / `curated/` / `spikesorted/` / `postprocessed/` from the results-collector layout + dispatch `job_*.json` + a synthetic `ecephys_*` session folder, runs `code/run_capsule.py`, and copies the resulting NWB (with a `units` table) into the single config's `coordinate_output_root`.

## Imports and prerequisites

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "neuroconv", "pynwb", "hdmf-zarr", "aind-nwb-utils",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Checked 4 packages in 17ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'neuroconv', 'pynwb', 'hdmf-zarr', 'aind-nwb-utils'], returncode=0)

## Build the scan config

In [3]:
nwb_input_path = (Path.cwd() / "output/10_nwb_results").resolve()
collected_output_path = (Path.cwd() / "output/07_collected_results").resolve()
dispatch_output_path = (Path.cwd() / "output/01_dispatch_results").resolve()
for p in (nwb_input_path, collected_output_path, dispatch_output_path):
    assert p.exists(), f"{p} not found. Run earlier notebooks first."

scan_config = obi.AINDUnitsNWBScanConfig(
    initialize=obi.AINDUnitsNWBScanConfig.Initialize(
        nwb_input_path=nwb_input_path,
        collected_output_path=collected_output_path,
        dispatch_output_path=dispatch_output_path,
    ),
)

## Generate the grid scan and run the units-NWB task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/aind_units_nwb/grid_scan",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 17:37:14,962] INFO: Cloning https://github.com/AllenNeuralDynamics/aind-units-nwb.git -> /tmp/aind-units-nwb


Cloning into '/tmp/aind-units-nwb'...


[2026-04-29 17:37:15,786] INFO: Patched utils.py for neuroconv API drift


[2026-04-29 17:37:16,019] INFO: Seeded data dir: /tmp/aind-units-nwb/data


[2026-04-29 17:37:16,020] INFO: Running python -u run_capsule.py


[2026-04-29 17:37:18,198] INFO: 

NWB EXPORT UNITS
Running NWB conversion with the following parameters:
Stub test: False
Stub units: 10
NWB backend: hdf5
Found 1 JSON job files
Found 1 processed recordings
Number of NWB files to write: 1
Number of streams to write for each file: 1
	Adding 10 units from block0_None_recording1
Added 1 streams
Writing time: 0.05s
Done writing ../results/session0_block0_recording1.nwb
NWB EXPORT UNITS time: 1.17s



[PosixPath('../../../obi-output/aind_units_nwb/grid_scan/AINDUnitsNWBSingleConfig')]

## Copy the per-coordinate NWB results next to the notebook

In [5]:
local_results_dir = Path.cwd() / "output/11_units_nwb_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
local_results_dir.mkdir(parents=True, exist_ok=True)

for single_config in grid_scan.single_configs:
    coord_dir = Path(single_config.coordinate_output_root)
    dest = local_results_dir / str(single_config.idx)
    dest.mkdir(parents=True, exist_ok=True)
    for entry in coord_dir.iterdir():
        target = dest / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

first = local_results_dir / "0"
if first.exists():
    for entry in first.iterdir():
        target = local_results_dir / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

for p in sorted(local_results_dir.iterdir()):
    print(" ", p.name)

  0
  obi_one_coordinate.json
  session0_block0_recording1.nwb


## Inspect the units table

In [6]:
from pynwb import NWBHDF5IO

nwb_files = sorted(local_results_dir.glob("*.nwb"))
print("NWB files:", [p.name for p in nwb_files])

if nwb_files:
    with NWBHDF5IO(str(nwb_files[0]), "r") as io:
        nwb = io.read()
        units = nwb.units
        if units is None:
            print("No units table written.")
        else:
            print("units columns:", list(units.colnames))
            print("num_units:    ", len(units.id))
            df = units.to_dataframe().reset_index()
            keep = [
                c
                for c in [
                    "id",
                    "unit_name",
                    "ks_unit_id",
                    "device_name",
                    "amplitude",
                    "depth",
                    "extremum_channel_index",
                ]
                if c in df.columns
            ]
            print()
            print(df[keep].to_string(index=False))

NWB files: ['session0_block0_recording1.nwb']
units columns: ['spike_times', 'electrodes', 'waveform_mean', 'waveform_sd', 'unit_name', 'sliding_rp_violation', 'exp_decay', 'trough_half_width', 'amplitude_median', 'drift_ptp', 'drift_std', 'waveform_baseline_flatness', 'peak_before_to_trough_ratio', 'amplitude', 'velocity_below', 'nn_hit_rate', 'main_peak_to_trough_ratio', 'depth', 'device_name', 'original_cluster_id', 'estimated_y', 'peak_after_to_trough_ratio', 'spread', 'estimated_x', 'silhouette', 'sync_spike_8', 'presence_ratio', 'nn_miss_rate', 'isolation_distance', 'd_prime', 'firing_range', 'extremum_channel_index', 'firing_rate', 'isi_violations_ratio', 'peak_before_to_peak_after_ratio', 'peak_to_trough_duration', 'rp_violations', 'shank', 'peak_after_width', 'peak_half_width', 'trough_width', 'l_ratio', 'num_negative_peaks', 'sync_spike_2', 'main_to_next_extremum_duration', 'amplitude_cv_range', 'num_spikes', 'recovery_slope', 'rp_contamination', 'sync_spike_4', 'velocity_abo